<!-- torchleet:colab -->
[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/Fengyee/TorchLeet/blob/main/torch/easy/cnn/CNN.ipynb)

Check your work: `pip install -e python/` once from the repo root (on Colab: `!pip install -q "git+https://github.com/Fengyee/TorchLeet#subdirectory=python"`), then `from torchleet import check; check("cnn", ...)`


# Problem: Implement a CNN for CIFAR-10 in PyTorch

### Problem Statement
You are tasked with implementing a **Convolutional Neural Network (CNN)** for image classification on the **CIFAR-10** dataset using PyTorch. The model should contain convolutional layers for feature extraction, batch normalization, pooling layers for downsampling, and fully connected layers for classification. Your goal is to complete the CNN model by defining the necessary layers and implementing the forward pass.

### Requirements
1. **Define the CNN Model** (`CNNModel`):
   - Stack **several convolution → batch norm → ReLU → max-pool stages**. Each 2×2 pool halves the spatial size: 32×32 → 16×16 → 8×8.
   - Flatten, then use **fully connected layers** to produce 10 logits, one per class. Do not apply a softmax: `nn.CrossEntropyLoss` expects raw logits.
   - The model must accept input of shape `(batch, 3, 32, 32)` and return `(batch, 10)`.
2. **Train and evaluate**: call `model.train()` before training and `model.eval()` before testing. BatchNorm normalizes with the current batch's statistics in training mode and with its running averages in eval mode, so forgetting `eval()` makes test accuracy depend on how the test set is batched.

### Constraints
- Use multiple convolutional and pooling stages before the fully connected layers. Pooling early keeps the first `nn.Linear` small. One pool followed by `Linear(64*16*16, 128)` puts 2M of the network's parameters in one layer that overfits quickly.
- CIFAR-10 has 10 classes.

<details>
  <summary>💡 Hint</summary>
  Track the shape through the network: <code>Conv2d(kernel_size=3, padding=1)</code> keeps H×W, and <code>MaxPool2d(2)</code> halves it. After two stages with 64 output channels, the first fully connected layer needs <code>in_features = 64 * 8 * 8</code>.
  <br>
  A convolution followed by BatchNorm does not need its own bias (<code>bias=False</code>), because BatchNorm's shift replaces it.
</details>

In [ ]:
import torch
import torch.nn as nn
import torch.optim as optim
import torchvision
import torchvision.transforms as transforms

In [ ]:
# Define the CNN Model
# TODO: Add convolutional, batch norm, pooling, and fully connected layers
class CNNModel(nn.Module):
    def __init__(self):
        super().__init__()
        ...

    def forward(self, x):
        ...

In [ ]:
# Load CIFAR-10 dataset
transform = transforms.Compose([
    transforms.ToTensor(),
    transforms.Normalize((0.5, 0.5, 0.5), (0.5, 0.5, 0.5))
])

train_dataset = torchvision.datasets.CIFAR10(root='./data', train=True, download=True, transform=transform)
train_loader = torch.utils.data.DataLoader(train_dataset, batch_size=64, shuffle=True)

test_dataset = torchvision.datasets.CIFAR10(root='./data', train=False, download=True, transform=transform)
test_loader = torch.utils.data.DataLoader(test_dataset, batch_size=64, shuffle=False)

In [ ]:
# Initialize the model, loss function, and optimizer
model = CNNModel()
criterion = nn.CrossEntropyLoss()
optimizer = optim.Adam(model.parameters(), lr=0.001)

# Training loop
epochs = 10
for epoch in range(epochs):
    model.train()  # BatchNorm: use batch statistics and update the running averages
    running = 0.0
    for images, labels in train_loader:
        # Forward pass
        outputs = model(images)
        loss = criterion(outputs, labels)

        # Backward pass and optimization
        optimizer.zero_grad()
        loss.backward()
        optimizer.step()
        running += loss.item() * images.size(0)

    print(f"Epoch [{epoch + 1}/{epochs}], Loss: {running / len(train_dataset):.4f}")

In [ ]:
# Evaluate on the test set
model.eval()  # BatchNorm: use the running averages collected during training
correct = 0
total = 0
with torch.no_grad():
    for images, labels in test_loader:
        outputs = model(images)
        predicted = outputs.argmax(dim=1)
        total += labels.size(0)
        correct += (predicted == labels).sum().item()

print(f"Test Accuracy: {100 * correct / total:.2f}%")